# Práctico 2 – System prompts que se prueban

**Magíster en Inteligencia Artificial UC · INF3880 – Inteligencia Artificial Agéntica**

**Profesores:** Vicente Domínguez, Antonio Ossa · **Ayudante:** Michelle Madrid

**Nombre del alumno:** ESCRIBIR NOMBRE AQUÍ

> Este práctico **no es evaluado**. Son ejercicios propuestos para trabajar en el Módulo 2 de la clase 2 (lunes 26/10/2026) y terminar en casa si hace falta. La versión con soluciones se publica después de la clase.

Compararemos tres versiones de un **system prompt** para clasificar correos de clientes, midiendo la exactitud con un set de casos, y forzaremos **salidas estructuradas** validadas con Pydantic, usando la **API gratuita de Gemini** como en el práctico 1. La idea central: los prompts se evalúan, no se adivinan.


## Configuración inicial

Usaremos la **API de Gemini en su nivel gratuito** (no requiere tarjeta de crédito). Cada alumno crea su propia key; toma unos 2 minutos.

### Paso A · Crear tu API key en Google AI Studio
1. Abre **https://aistudio.google.com/apikey** e inicia sesión con tu cuenta de Google (personal; la cuenta institucional puede tener AI Studio bloqueado).
2. La primera vez te pedirá **aceptar los términos de servicio** de la Gemini API. Al aceptarlos, Google crea automáticamente un proyecto por defecto.
3. Haz clic en **"Create API key"** (si ya tienes un proyecto, elige "Create API key in existing project" o el proyecto por defecto).
4. Copia la key con el botón **"Copiar clave"** (las keys actuales empiezan con `AQ.`; las antiguas con `AIza`). Guárdala en un lugar seguro: la vas a necesitar en el paso B. Si la pierdes, puedes crear otra desde la misma página.
5. Opcional pero recomendado: en la lista de keys, pasa el mouse sobre la etiqueta **"Unrestricted"** → **"Add restrictions"** → **"Restrict to Gemini API only"** → **"Restrict key"**.

> ¿Dónde ver los límites y el uso? En AI Studio, menú lateral → **Dashboard** → **Usage & Billing** (o https://aistudio.google.com/usage). Ahí aparece el plan "Free" y las cuotas por modelo.
>
> ¿Problemas? Si el botón "Create API key" no aparece, suele ser porque la cuenta es de una organización (Workspace) que restringe AI Studio: usa una cuenta personal de Gmail. Documentación oficial: https://ai.google.dev/gemini-api/docs/api-key

### Paso B · Guardar la key en Secrets de Colab
1. En este notebook, haz clic en el ícono de la **llave 🔑** de la barra lateral izquierda ("Secrets").
2. **"+ Add new secret"**: en *Name* escribe exactamente `GEMINI_API_KEY`; en *Value* pega la key.
3. Activa el interruptor **"Notebook access"** para este notebook.
4. **Nunca pegues la key en una celda, ni la subas a un repositorio, ni la compartas.** Trátala como una contraseña: si se filtra, crea una nueva y borra la antigua en AI Studio.

### Paso C · Instalar y configurar
Instalamos la librería `google-genai` y definimos el modelo en una sola variable `MODEL` para poder cambiarlo sin tocar el resto del notebook.

> ⚠️ **Dos cosas que debes saber del nivel gratuito.** (1) Según los términos de Google, los prompts y respuestas del nivel sin pago **pueden usarse para mejorar sus productos y ser revisados por personas**: no envíes datos personales, confidenciales ni de tu empresa. (2) Tiene **límites de uso por minuto y por día**; si recibes un error 429 (*resource exhausted*), espera un momento y reintenta. Los límites de tu proyecto se ven en AI Studio.

In [1]:
%%capture
!pip install -q --upgrade google-genai pandas pydantic "bottleneck>=1.5.0"

In [2]:
import os, time
from google.colab import userdata

# 1) Leer la key desde Secrets, con mensajes claros si algo falta
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except userdata.SecretNotFoundError:
    raise SystemExit("No existe un secreto llamado GEMINI_API_KEY. Créalo en el panel de Secrets (ícono de la llave) con ese nombre exacto.")
except userdata.NotebookAccessError:
    raise SystemExit("El secreto GEMINI_API_KEY existe pero no tiene 'Notebook access' activado para este notebook.")
assert GEMINI_API_KEY and len(GEMINI_API_KEY) > 20, \
    "El secreto GEMINI_API_KEY está vacío. Pega la key completa desde https://aistudio.google.com/apikey (botón 'Copiar clave')."
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# 2) Elegir el modelo: buscamos uno que esté disponible en TU proyecto gratuito y que respete `temperature`
#    (los modelos Gemini 3.6+ y 4 la ignoran, por eso preferimos Flash-Lite 3.5 o 2.5)
from google import genai
client = genai.Client()
PREFERIDOS = ["gemini-3.5-flash-lite", "gemini-2.5-flash-lite", "gemini-3.1-flash-lite", "gemini-2.5-flash", "gemini-3.8-flash"]
disponibles = []
for m in client.models.list():
    acciones = getattr(m, "supported_actions", None) or []
    if not acciones or "generateContent" in acciones:
        disponibles.append(m.name.replace("models/", ""))
MODEL = next((p for p in PREFERIDOS if p in disponibles), None)
if MODEL is None:
    print("Ninguno de los modelos preferidos está disponible. Modelos de tu proyecto:", disponibles[:30])
    MODEL = next((d for d in disponibles if "flash" in d), disponibles[0])
RESPALDO = [p for p in PREFERIDOS if p in disponibles and p != MODEL]   # por si el modelo elegido falla (error 500/503)
print("Modelo elegido:", MODEL, "| respaldo:", RESPALDO)

# Precios de referencia del nivel DE PAGO, en USD por 1M tokens (https://ai.google.dev/gemini-api/docs/pricing).
# En el nivel gratuito no pagas, pero calcularemos cuánto costaría para entender la economía de un LLM.
PRECIOS = {
    "gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50},
    "gemini-3.8-flash":      {"input": 0.75, "output": 3.75},
    "gemini-3.1-flash-lite": {"input": 0.25, "output": 1.50},
    "gemini-2.5-flash":      {"input": 0.30, "output": 2.50},
    "gemini-2.5-flash-lite": {"input": 0.10, "output": 0.40},
}

def costo_usd(input_tokens, output_tokens, model=None):
    p = PRECIOS.get(model or MODEL, PRECIOS["gemini-3.5-flash-lite"])   # si el modelo no está en la tabla, usa un precio de referencia
    return ((input_tokens or 0) * p["input"] + (output_tokens or 0) * p["output"]) / 1_000_000

print("Listo. Modelo:", MODEL)

Modelo elegido: gemini-3.5-flash-lite | respaldo: ['gemini-2.5-flash-lite', 'gemini-3.1-flash-lite', 'gemini-2.5-flash', 'gemini-3.8-flash']
Listo. Modelo: gemini-3.5-flash-lite


Verificamos que la key funciona:

In [3]:
from google.genai import types

def generar(contents, reintentos=3, **config):
    """Llama al modelo con manejo de los errores típicos del nivel gratuito:
    - 429 (límite de uso): espera y reintenta.
    - 500/503 (servidor saturado o modelo no disponible en ese momento): reintenta con esperas crecientes
      y, si sigue fallando, cambia al siguiente modelo de RESPALDO."""
    global MODEL
    for intento in range(reintentos + 1):
        try:
            return client.models.generate_content(model=MODEL, contents=contents,
                                                  config=types.GenerateContentConfig(**config))
        except Exception as e:
            msg = str(e)
            if ("429" in msg or "RESOURCE_EXHAUSTED" in msg) and intento < reintentos:
                print("  (límite de uso alcanzado, esperando 20 s...)"); time.sleep(20)
            elif any(k in msg for k in ("500", "503", "INTERNAL", "UNAVAILABLE", "overloaded", "not available")):
                if intento < reintentos:
                    espera = 5 * 2 ** intento          # 5, 10, 20 s
                    print(f"  (error transitorio del servidor, reintentando en {espera} s...)"); time.sleep(espera)
                elif RESPALDO:
                    anterior, MODEL = MODEL, RESPALDO.pop(0)
                    print(f"  ⚠️ {anterior} sigue fallando; cambiamos a {MODEL} y seguimos.")
                    return generar(contents, reintentos=reintentos, **config)
                else:
                    raise SystemExit("Gemini devolvió errores 500/503 en todos los modelos disponibles. Suele ser saturación temporal: espera unos minutos y vuelve a ejecutar esta celda.")
            elif "404" in msg or "NOT_FOUND" in msg:
                raise SystemExit(f"El modelo {MODEL} no está disponible en tu proyecto. Cambia MODEL por otro de la lista de la celda anterior.")
            elif "403" in msg or "401" in msg or "API_KEY_INVALID" in msg:
                raise SystemExit("La API key fue rechazada. Verifica que la copiaste completa desde https://aistudio.google.com/apikey y que no está restringida a otra API.")
            else:
                raise

print(generar("Responde solo: OK").text)

OK


## Parte guiada 1 · El problema y los datos

Andina Cargo recibe correos de clientes y quiere **clasificarlos** en `facturacion`, `tecnico`, `reclamo` u `otro` para derivarlos al equipo correcto. Tenemos 31 correos ya etiquetados a mano: nuestro **set de casos de prueba**. Los 15 primeros son claros; los 16 siguientes son **difíciles** a propósito: sarcasmo, palabras clave engañosas, dos temas en un correo, otro idioma, correos sin contenido. Así es un set real.

> Al etiquetar casos ambiguos tuvimos que tomar decisiones (por ejemplo: "una entrega que no llegó es reclamo, aunque esté escrita como pregunta"). Un set de prueba **codifica esas decisiones**; si no estás de acuerdo con alguna etiqueta, eso también es parte del método: discútela y cámbiala.

In [4]:
import warnings
warnings.filterwarnings("ignore", message="Pandas requires version")
import pandas as pd


CASOS = [
 # --- claros ---
 ("La factura 7781 tiene el RUT equivocado, ¿pueden reemitirla?", "facturacion", "facil"),
 ("No me llega el correo de confirmación cuando creo un pedido en la plataforma.", "tecnico", "facil"),
 ("El pedido P-1004 llegó con dos cajas rotas y nadie me avisó del retraso.", "reclamo", "facil"),
 ("¿Tienen cobertura a Punta Arenas?", "otro", "facil"),
 ("Pagué dos veces el mismo pedido, necesito la devolución.", "facturacion", "facil"),
 ("La API devuelve error 500 al consultar el estado de un pedido.", "tecnico", "facil"),
 ("Llevo tres días esperando respuesta de su ejecutiva, pésimo servicio.", "reclamo", "facil"),
 ("Quisiera una cotización para 200 pallets mensuales.", "otro", "facil"),
 ("¿Pueden enviar la factura en formato electrónico al nuevo correo de contabilidad?", "facturacion", "facil"),
 ("La app móvil se cierra sola al abrir el tracking.", "tecnico", "facil"),
 ("El transportista fue grosero con nuestro bodeguero.", "reclamo", "facil"),
 ("¿Cuál es el horario de recepción en la bodega de Santiago?", "otro", "facil"),
 ("El monto de la nota de crédito no coincide con lo acordado.", "facturacion", "facil"),
 ("No puedo restablecer mi contraseña, el link expira de inmediato.", "tecnico", "facil"),
 ("Mi pedido dice entregado pero nunca llegó.", "reclamo", "facil"),
 # --- difíciles: palabra clave engañosa, sarcasmo, dos temas, otro idioma, sin contenido ---
 ("Gracias por resolver tan rápido el tema de la factura, quedó perfecta.", "otro", "dificil"),            # elogio con la palabra "factura"
 ("Qué maravilla, tercer despacho atrasado este mes. Felicitaciones al equipo.", "reclamo", "dificil"),     # sarcasmo
 ("El tracking dice 'entregado' desde ayer, pero nuestra bodega estaba cerrada. ¿Dónde lo dejaron?", "reclamo", "dificil"),  # pregunta, pero entrega no realizada
 ("¿Me pueden explicar por qué la factura de octubre viene con un recargo por 'manejo especial'?", "facturacion", "dificil"),  # pregunta sobre factura, no reclamo
 ("Necesito integrar su API con nuestro ERP, ¿tienen documentación o un ejecutivo técnico?", "otro", "dificil"),  # "API" pero es consulta comercial, no hay error
 ("Mi ejecutiva dijo que me llamaría el lunes por la nota de crédito y nunca llamó.", "reclamo", "dificil"),   # molestia por servicio, aunque mencione nota de crédito
 ("Hola, buenas tardes.", "otro", "dificil"),                                                                   # sin contenido
 ("El sistema muestra el P-1002 como 'en preparación' pero ya lo recibí. ¿Es un error de la plataforma?", "tecnico", "dificil"),  # error de datos, sin daño
 ("Good morning, could you send me the invoice for order P-1001 in English?", "facturacion", "dificil"),       # otro idioma
 ("Dejé de recibir notificaciones desde que cambié mi contraseña; además la última factura tiene mal la dirección.", "tecnico", "dificil"),  # dos temas: gana el primero
 ("¿Pueden cotizar 50 pallets a Punta Arenas? La última vez el precio me pareció alto.", "otro", "dificil"),   # cotización con queja leve
 ("Se me olvidó la clave del portal, pero no es urgente, cuando puedan.", "tecnico", "dificil"),               # tono amable, sigue siendo técnico
 ("Adjunto la foto de las cajas mojadas que llegaron hoy.", "reclamo", "dificil"),                             # reclamo implícito
 ("¿El reembolso del pedido duplicado ya fue procesado? Me dijeron 5 días hábiles y van 8.", "facturacion", "dificil"),  # pide dinero → facturación, aunque haya demora
 ("O pedido chegou com a embalagem rasgada e faltando duas caixas.", "reclamo", "dificil"),                    # portugués
 ("asdfgh 1234 prueba", "otro", "dificil"),                                                                    # basura
]
df = pd.DataFrame(CASOS, columns=["correo", "esperado", "dificultad"])
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
conteo = df.dificultad.value_counts()
print(f"Casos: {len(df)} | faciles: {int(conteo.get('facil', 0))} | dificiles: {int(conteo.get('dificil', 0))}")
df

Casos: 31 | faciles: 15 | dificiles: 16


,correo,esperado,dificultad
0,"La factura 7781 tiene el RUT equivocado, ¿pued...",facturacion,facil
1,No me llega el correo de confirmación cuando c...,tecnico,facil
2,El pedido P-1004 llegó con dos cajas rotas y n...,reclamo,facil
3,¿Tienen cobertura a Punta Arenas?,otro,facil
4,"Pagué dos veces el mismo pedido, necesito la d...",facturacion,facil
5,La API devuelve error 500 al consultar el esta...,tecnico,facil
6,Llevo tres días esperando respuesta de su ejec...,reclamo,facil
7,Quisiera una cotización para 200 pallets mensu...,otro,facil
8,¿Pueden enviar la factura en formato electróni...,facturacion,facil
9,La app móvil se cierra sola al abrir el tracking.,tecnico,facil


## Parte guiada 2 · Tres versiones de system prompt

Vamos a comparar un prompt vago, uno estructurado y uno con ejemplos. Para que la comparación sea justa, el resto (modelo, temperature, correos) no cambia.

> **Cómo evaluamos sin chocar con el límite del nivel gratuito.** Clasificar los 31 correos uno por uno serían 31 llamadas por versión y el nivel gratuito limita las llamadas por minuto. En vez de eso, enviamos **todos los correos numerados en una sola llamada** y pedimos la respuesta como una lista JSON con `{id, categoria}` (salida estructurada). Resultado: 1 llamada por versión, segundos en vez de minutos. Es una técnica real ("batch prompting") que además te muestra el valor del esquema: sin él, alinear 31 respuestas sería frágil.

In [5]:
PROMPTS = {
 "v1_vago": "Clasifica el correo.",

 "v2_estructurado": """Eres el clasificador de correos de Andina Cargo (logística).
Tarea: asignar UNA categoría al correo del cliente.
Categorías:
- facturacion: facturas, pagos, notas de crédito, datos tributarios.
- tecnico: errores de la plataforma web, app o API, accesos y contraseñas.
- reclamo: quejas por daños, retrasos, mal servicio o entregas no realizadas.
- otro: consultas comerciales, cotizaciones, horarios, cobertura.
Responde SOLO con el nombre de la categoría, en minúsculas, sin explicación.""",

 "v3_con_ejemplos": None,   # lo construimos abajo a partir de v2
}
PROMPTS["v3_con_ejemplos"] = PROMPTS["v2_estructurado"] + """

Ejemplos:
Correo: "La boleta salió con un monto distinto al cotizado" -> facturacion
Correo: "La página de tracking no carga desde ayer" -> tecnico
Correo: "La carga llegó mojada" -> reclamo
Correo: "¿Hacen despachos los sábados?" -> otro"""

In [6]:
from pydantic import BaseModel
PAUSA = 2.0   # segundos entre llamadas, para respetar el límite por minuto del nivel gratuito

class Etiqueta(BaseModel):
    id: int
    categoria: str      # str a propósito: queremos VER si un prompt vago inventa categorías

class Lote(BaseModel):
    resultados: list[Etiqueta]

def clasificar_lote(correos, system_prompt):
    """Clasifica todos los correos en UNA llamada. Devuelve una lista alineada con `correos`."""
    lista = "\n".join(f"{i}. {c}" for i, c in enumerate(correos))
    contenido = f"Clasifica cada uno de los siguientes correos, identificados por su número:\n\n{lista}"
    r = generar(contenido, system_instruction=system_prompt, temperature=0,
                response_mime_type="application/json", response_schema=Lote)
    time.sleep(PAUSA)
    lote = r.parsed if getattr(r, "parsed", None) is not None else Lote.model_validate_json(r.text)
    por_id = {e.id: e.categoria.strip().lower().strip(".") for e in lote.resultados}
    return [por_id.get(i, "sin_respuesta") for i in range(len(correos))]

REPETICIONES = 3    # el modelo no es determinista ni a temperature=0: evaluamos cada versión varias veces
CORRIDAS = {}       # nombre -> lista de listas de predicciones (una por corrida)

def evaluar(system_prompt, nombre, repeticiones=REPETICIONES):
    accs = []
    CORRIDAS[nombre] = []
    for k in range(repeticiones):
        preds = clasificar_lote(list(df["correo"]), system_prompt)
        CORRIDAS[nombre].append(preds)
        accs.append((pd.Series(preds) == df["esperado"]).mean())
    df[nombre] = CORRIDAS[nombre][-1]                 # guardamos la última corrida para mirar los casos
    ok = df[nombre] == df["esperado"]
    acc_f, acc_d = ok[df.dificultad == "facil"].mean(), ok[df.dificultad == "dificil"].mean()
    print(f"{nombre:18} exactitud promedio {sum(accs)/len(accs):.2f}  (rango {min(accs):.2f}–{max(accs):.2f} en {repeticiones} corridas) | última corrida: fáciles {acc_f:.2f}, difíciles {acc_d:.2f}")
    return sum(accs) / len(accs)

resultados = {nombre: evaluar(p, nombre) for nombre, p in PROMPTS.items()}
pd.Series(resultados, name="exactitud_promedio")

v1_vago            exactitud promedio 0.08  (rango 0.00–0.23 en 3 corridas) | última corrida: fáciles 0.00, difíciles 0.00
v2_estructurado    exactitud promedio 0.90  (rango 0.90–0.90 en 3 corridas) | última corrida: fáciles 1.00, difíciles 0.81
v3_con_ejemplos    exactitud promedio 0.91  (rango 0.90–0.94 en 3 corridas) | última corrida: fáciles 1.00, difíciles 0.88


v1_vago            0.075269
v2_estructurado    0.903226
v3_con_ejemplos    0.913978
Name: exactitud_promedio, dtype: float64

💡 **Observa el rango.** Con 31 casos, **un caso equivale a 3 puntos de exactitud**, y el modelo cambia alguna etiqueta entre corridas aunque `temperature=0`. Por eso una diferencia de 1 o 2 casos entre dos versiones no significa que una sea mejor: está dentro del ruido. Para concluir "v3 es mejor que v2" necesitas o más casos, o varias corridas, o idealmente ambas. Esta es la respuesta práctica a "¿cómo sabrías que tu prompt nuevo es mejor?".

In [7]:
# ¿Dónde falla cada versión? Comparamos v2 y v3 (v1 falla en todo porque inventa los nombres de las categorías).
versiones = [v for v in PROMPTS if v != "v1_vago"]
cols = ["correo", "esperado", "dificultad"] + versiones
fallas = df[(df[versiones].ne(df["esperado"], axis=0)).any(axis=1)]
fallas[cols]

,correo,esperado,dificultad,v2_estructurado,v3_con_ejemplos
15,Gracias por resolver tan rápido el tema de la ...,otro,dificil,facturacion,facturacion
19,"Necesito integrar su API con nuestro ERP, ¿tie...",otro,dificil,tecnico,otro
24,Dejé de recibir notificaciones desde que cambi...,tecnico,dificil,facturacion,facturacion


In [8]:
# Casos "inestables": cambiaron de etiqueta entre corridas de la misma versión
for nombre in versiones:
    corridas = CORRIDAS[nombre]
    inestables = [i for i in range(len(df)) if len({c[i] for c in corridas}) > 1]
    print(f"{nombre}: {len(inestables)} caso(s) inestable(s)")
    for i in inestables:
        etiquetas = " / ".join(c[i] for c in corridas)
        print(f"   · [{df.dificultad[i]}] {df.correo[i][:60]}…  → {etiquetas}  (esperado: {df.esperado[i]})")

v2_estructurado: 0 caso(s) inestable(s)
v3_con_ejemplos: 1 caso(s) inestable(s)
   · [dificil] Necesito integrar su API con nuestro ERP, ¿tienen documentac…  → tecnico / tecnico / otro  (esperado: otro)


💡 **Observa:** en los casos fáciles todas las versiones aciertan; la diferencia está en los **difíciles**. La versión vaga devuelve nombres inventados ("soporte técnico", "facturación" con tilde): semánticamente razonables, inútiles para un sistema; ese es el problema de formato. La estructurada cae en las trampas de palabra clave (la "factura" del elogio, la "API" de la consulta comercial); los ejemplos ayudan en algunos casos ambiguos y en otros no. Los casos inestables suelen ser justamente los ambiguos: si una etiqueta "esperada" te parece discutible, el set de prueba es una decisión de negocio escrita y se puede cambiar.

## Parte guiada 3 · Salidas estructuradas con Pydantic

Pedir "solo la categoría" es frágil. Es más robusto **declarar el esquema** de la respuesta: Gemini acepta un modelo Pydantic como `response_schema` y devuelve JSON que cumple ese esquema; el SDK lo entrega ya convertido en `response.parsed`.

In [9]:
from pydantic import BaseModel
from typing import Literal

class Clasificacion(BaseModel):
    categoria: Literal["facturacion", "tecnico", "reclamo", "otro"]
    urgencia: Literal["baja", "media", "alta"]
    resumen: str   # una frase

def generar_estructurado(contents, esquema, **config):
    """Pide al modelo una respuesta que cumpla el esquema Pydantic y devuelve el objeto validado."""
    r = generar(contents, response_mime_type="application/json", response_schema=esquema, **config)
    time.sleep(PAUSA)
    return r.parsed if getattr(r, "parsed", None) is not None else esquema.model_validate_json(r.text)

def clasificar_estructurado(correo):
    return generar_estructurado(correo, Clasificacion, temperature=0,
        system_instruction=PROMPTS["v2_estructurado"] + "\nAdemás estima la urgencia y resume el correo en una frase.")

for c in df["correo"][:3]:      # 3 llamadas individuales, una por correo
    print(clasificar_estructurado(c))

categoria='facturacion' urgencia='media' resumen='Solicitud de reemisión de la factura 7781 por error en el RUT.'
categoria='tecnico' urgencia='media' resumen='El cliente reporta que no recibe el correo de confirmación al crear un pedido en la plataforma.'
categoria='reclamo' urgencia='alta' resumen='El cliente reporta daños en la mercancía del pedido P-1004 y retraso en la entrega sin previo aviso.'


💡 **Observa:** `parsed` es un objeto Python validado; si el modelo devolviera una categoría fuera de la lista, Pydantic lanza un error en vez de contaminar tu sistema. Esto será la base de las *tools* en la clase 3.

## Ejercicios propuestos

De dificultad creciente. El último siempre es abierto. Si te trabas, revisa las pistas y la parte guiada; la solución se publica después de la clase.

### Ejercicio 1 – Subir la exactitud en los casos difíciles

Escribe una `v4` del prompt que corrija los errores que viste en la tabla de fallas: agrega **reglas de desempate** explícitas (¿qué gana cuando hay dos temas? ¿qué pasa con un elogio que menciona una factura? ¿y un correo en otro idioma o sin contenido?) y, si hace falta, ejemplos de los casos trampa. Meta: ≥ 90 % en los difíciles sin bajar en los fáciles, **de forma estable** (el rango de las 3 corridas también debe mejorar). Anota qué cambio tuvo más impacto y qué casos siguen fallando: ¿es culpa del prompt o la etiqueta es discutible?

In [10]:
PROMPTS["v4_mejorado"] = """TODO: tu prompt"""
evaluar(PROMPTS["v4_mejorado"], "v4_mejorado")
df[df["v4_mejorado"] != df["esperado"]][["correo", "esperado", "dificultad", "v4_mejorado"]]

v4_mejorado        exactitud promedio 0.00  (rango 0.00–0.00 en 3 corridas) | última corrida: fáciles 0.00, difíciles 0.00


,correo,esperado,dificultad,v4_mejorado
0,"La factura 7781 tiene el RUT equivocado, ¿pued...",facturacion,facil,facturación
1,No me llega el correo de confirmación cuando c...,tecnico,facil,soporte técnico
2,El pedido P-1004 llegó con dos cajas rotas y n...,reclamo,facil,logística y despacho
3,¿Tienen cobertura a Punta Arenas?,otro,facil,comercial / ventas
4,"Pagué dos veces el mismo pedido, necesito la d...",facturacion,facil,facturación
5,La API devuelve error 500 al consultar el esta...,tecnico,facil,soporte técnico
6,Llevo tres días esperando respuesta de su ejec...,reclamo,facil,atención al cliente
7,Quisiera una cotización para 200 pallets mensu...,otro,facil,comercial / ventas
8,¿Pueden enviar la factura en formato electróni...,facturacion,facil,facturación
9,La app móvil se cierra sola al abrir el tracking.,tecnico,facil,soporte técnico


### Ejercicio 2 – ¿Ayuda el razonamiento paso a paso?

Crea una variante que pida al modelo **razonar brevemente antes** de dar la categoría (en el esquema del lote, cada etiqueta con campos `id`, `razonamiento` y `categoria`, en ese orden). Compara exactitud, tokens de salida y costo estimado contra la v2. ¿Vale la pena?

Para que quiere un nuevo campo en la estructura de Lote (razonamiento), si el razonamiento es un atributo de configuración que se le pasa al modelo junto al contexto ???  (Personalmente no lo necesito)

Conclusión:
Con razonamineto el modelo de demora mas ms en contestar, usa mas tokens (es mas costoso) y pierdo exactitud


In [ ]:
class EtiquetaCoT(BaseModel):
    id: int
    razonamiento: str
    categoria: Literal["facturacion", "tecnico", "reclamo", "otro"]

class LoteCoT(BaseModel):
    resultados: list[EtiquetaCoT]

# TODO: adapta clasificar_lote para usar LoteCoT, calcula la exactitud y lee usage_metadata.candidates_token_count


### Ejercicio 3 – Extracción estructurada de un pedido

Define un modelo Pydantic `Pedido` con `sku`, `cantidad`, `ciudad` (opcionales si faltan) y una lista `datos_faltantes`. Prueba con 3 frases, una incompleta. Compara con el ejercicio 3 del práctico 1: ¿qué ganaste con el esquema?

In [12]:
from typing import Optional

class Pedido(BaseModel):
    # TODO
    pass


### Ejercicio 4 – Abierto: casos de prueba de tu trabajo

Arma un set de 10 casos de una tarea de clasificación o extracción de tu área (anonimizados: es el nivel gratuito), incluyendo al menos 3 casos difíciles. Escribe dos versiones de prompt y mide cuál funciona mejor con `clasificar_lote`.

In [13]:
MIS_CASOS = [("texto", "etiqueta"), ]
# TODO


## Para seguir en casa

- Tu key gratuita de Gemini sigue funcionando fuera de clase: los límites diarios se reinician cada día (medianoche hora del Pacífico). Puedes ver el uso y los límites de tu proyecto en [AI Studio](https://aistudio.google.com/).
- Si quieres probar otro modelo, cambia `MODEL` (por ejemplo a `"gemini-3.8-flash"`), pero recuerda que los modelos 3.6+ ignoran `temperature`.
- Recuerda la regla del nivel gratuito: nada de datos personales ni confidenciales en los prompts.
- **Lectura relacionada:** Lanham (2024), capítulo sobre prompt engineering.
